
# Local RAG Demo (Jupyter) — Ollama Embeddings + FAISS


Try these in a terminal:
```bash
ollama list
ollama pull nomic-embed-text
ollama pull gemma:2b
```


## Ollama Quickstart — Chat & Embeddings

[Ollama](https://ollama.com) runs LLMs locally. It exposes a small HTTP API on `http://localhost:11434`, and the official `ollama` Python package wraps that API.

There are two ways to call it from Python:

```python
import ollama

# (1) Vanilla — uses the module-level client (reads env vars like HTTP(S)_PROXY)
ollama.chat(model="gemma:2b", messages=[{"role": "user", "content": "hi"}])
ollama.embeddings(model="embeddinggemma:latest", prompt="hello world")

# (2) Explicit Client — pin the host and ignore proxy env vars
client = ollama.Client(host="http://localhost:11434", trust_env=False)
client.chat(model="gemma:2b", messages=[{"role": "user", "content": "hi"}])
client.embeddings(model="embeddinggemma:latest", prompt="hello world")
```

### Why `trust_env=False`?

When a **VPN / corporate proxy** is active, environment variables like `HTTP_PROXY`, `HTTPS_PROXY`, or `ALL_PROXY` are set globally. The default `ollama` client (built on top of `httpx`) will honor those proxies and try to route the `localhost:11434` request **through the VPN**, which then fails (the proxy can't see your local Ollama).

`trust_env=False` tells the underlying `httpx` client to **ignore the proxy env vars** so the request goes straight to `localhost`. That's why everywhere below we use:

```python
ollama.Client(host="http://localhost:11434", trust_env=False)
```

instead of the bare `ollama.chat(...)` / `ollama.embeddings(...)` helpers.

### Prereqs

In a terminal:
```bash
ollama list                       # check installed models
ollama pull gemma:2b              # small chat model
ollama pull embeddinggemma        # embedding model used in this notebook
# alternative embedding model:
# ollama pull nomic-embed-text
```


In [27]:
# Install the Ollama Python client (safe to re-run).
%pip -q install ollama


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [28]:
import os
from dotenv import load_dotenv

import textwrap



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

In [29]:
# === Demo 1: Chat with an Ollama model ===
# Using the explicit Client so the request bypasses any VPN/proxy env vars and
# goes straight to the local Ollama daemon.

import ollama

CHAT_MODEL = "gemma3:1b"   # change to whatever you've `ollama pull`-ed

client = ollama.Client(host="http://localhost:11434", trust_env=False)

resp = client.chat(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user",   "content": "Why did people vote for Donald Trump?"},
    ],
)

# The response is a dict-like object; the assistant text lives at resp["message"]["content"]
pretty_print("Model:   ", resp.get("model"))
pretty_print("Reply:   ", resp["message"]["content"])
pretty_print("Tokens:  ", resp.get("eval_count"), "(eval) /", resp.get("prompt_eval_count"), "(prompt)")

Model:    gemma3:1b
Reply:    Okay, let’s break down why people voted for Donald Trump. It’s a
really complex mix of factors, but here's a breakdown of some of the key
drivers:  **1. Economic Concerns:**  * **Job Losses:** Many felt his policies
led to job losses, particularly in manufacturing. * **Trade Imbalances:**  A
perception that the US was being unfairly treated in trade deals was a
significant concern. * **Globalization:** Some voters were wary of the
increasing influence of international institutions and trade agreements.  **2.
Cultural Issues & Identity:**  * **“Culture War”:**  A feeling that the
political landscape was increasingly divided along ideological lines, with a
focus on culture and identity. * **Nationalism:** A desire to reconnect with
American values and a sense of national pride. * **Immigration:**  His rhetoric
on immigration – particularly regarding border security – resonated with some
voters, though this was a controversial aspect.  **3. Political & Policy

In [30]:
# === Demo 2: Get an embedding vector from Ollama ===
# Same Client pattern (host pinned, env-proxies ignored) — this is exactly what
# search_windowed() and the chunk-embedding loop below use under the hood.

import numpy as np
import ollama

EMBED_MODEL_DEMO = "embeddinggemma:latest"   # or "nomic-embed-text"

client = ollama.Client(host="http://localhost:11434", trust_env=False)

resp = client.embeddings(
    model=EMBED_MODEL_DEMO,
    prompt="The quick brown fox jumps over the lazy dog.",
)

vec = np.asarray(resp["embedding"], dtype="float32")
pretty_print("Embedding model:  ", EMBED_MODEL_DEMO)
pretty_print("Vector dimension: ", vec.shape)         # e.g. (768,) for embeddinggemma, (384,) for nomic-embed-text
pretty_print("First 8 dims:     ", np.round(vec[:8], 4).tolist())
pretty_print("L2 norm:          ", float(np.linalg.norm(vec)))

# Quick cosine-similarity sanity check between two semantically related sentences.
def embed(text: str) -> np.ndarray:
    v = np.asarray(
        client.embeddings(model=EMBED_MODEL_DEMO, prompt=text)["embedding"],
        dtype="float32",
    )
    return v / (np.linalg.norm(v) + 1e-12)   # L2-normalize so dot == cosine

a = embed("A dog chases a cat in the garden.")
b = embed("In the yard, a puppy is running after a kitten.")
c = embed("The Fourier transform decomposes a signal into frequencies.")

pretty_print(f"\ncos(a, b) similar     = {float(a @ b):.3f}")
pretty_print(f"cos(a, c) unrelated   = {float(a @ c):.3f}")

Embedding model:   embeddinggemma:latest
Vector dimension:  (768,)
First 8 dims:      [-0.11029999703168869, 0.05389999970793724,
0.06880000233650208, -0.022299999371170998, -0.08060000091791153,
0.00419999985024333, 0.03519999980926514, 0.051600001752376556]
L2 norm:           1.0
 cos(a, b) similar     = 0.787
cos(a, c) unrelated   = 0.217


In [31]:

# If needed, install dependencies. (Safe to re-run.)
%pip -q install  faiss-cpu numpy pandas tqdm requests



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [32]:

# === Configuration (change these as needed) ===
EMBED_MODEL = "embeddinggemma:latest"   
#EMBED_MODEL = "nomic-embed-text"

WORDS_PER_CHUNK = 300
OVERLAP_WORDS   = 60
TOPK            = 5

# Toggle downloads off if you want to stay strictly offline (then we'll use tiny built-in samples).
DOWNLOAD_FROM_WEB = True

# A small selection of long classics (public domain). Even 2–3 will give you 300+ chunks.

GUTENBERG_BOOKS = {
    "Moby-Dick": "https://www.gutenberg.org/files/2701/2701-0.txt",
    "Pride and Prejudice": "https://www.gutenberg.org/files/1342/1342-0.txt",
    "Frankenstein": "https://www.gutenberg.org/files/84/84-0.txt",
    "Alice in Wonderland": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "Dracula": "https://www.gutenberg.org/files/345/345-0.txt",
    "A Tale of Two Cities": "https://www.gutenberg.org/files/98/98-0.txt",
    "The Great Gatsby": "https://www.gutenberg.org/cache/epub/64317/pg64317.txt",
    "Adventures of Sherlock Holmes": "https://www.gutenberg.org/files/1661/1661-0.txt",
    "War and Peace": "https://www.gutenberg.org/files/2600/2600-0.txt",
    "Jane Eyre": "https://www.gutenberg.org/files/1260/1260-0.txt",
    "The Picture of Dorian Gray": "https://www.gutenberg.org/files/174/174-0.txt",
    "Crime and Punishment": "https://www.gutenberg.org/files/2554/2554-0.txt",
    "Wuthering Heights": "https://www.gutenberg.org/files/768/768-0.txt",

}


GUTENBERG = [
    (title, url) for title, url in GUTENBERG_BOOKS.items()
]

CORPUS_DIR = "corpus_jupyter"


In [33]:

import os, re, json, textwrap
from pathlib import Path
import requests
import numpy as np
import pandas as pd
from tqdm import tqdm

import ollama
import faiss

import truststore
truststore.inject_into_ssl()

# Minimal helper: strip Project Gutenberg boilerplate if found.
START_MARK = re.compile(r"\*\*\* START OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)
END_MARK   = re.compile(r"\*\*\* END OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)

def strip_gutenberg_boilerplate(txt: str) -> str:
    start = START_MARK.search(txt)
    end = END_MARK.search(txt)
    if start and end and end.start() > start.end():
        return txt[start.end():end.start()].strip()
    # Fallback heuristics
    txt = re.sub(r"(?s)^.*?Project Gutenberg.*?eBook.*?\n", "", txt, flags=re.I)
    txt = re.sub(r"(?s)End of Project Gutenberg.*$", "", txt, flags=re.I)
    return txt.strip()

def l2_normalize(mat: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12
    return mat / norms


In [34]:
# === Check if pre-built artifacts exist; if so, load and skip heavy work ===
import json as _json

ARTIFACTS_DIR = "rag_artifacts"
_artifact_files = ["chunks.json", "embeddings.npy", "faiss_index.bin", "config.json"]
ARTIFACTS_LOADED = all((Path(ARTIFACTS_DIR) / f).exists() for f in _artifact_files)

if ARTIFACTS_LOADED:
    pretty_print("✅ Found pre-built artifacts! Loading from disk …")
    with open(Path(ARTIFACTS_DIR) / "chunks.json", encoding="utf-8") as _f:
        chunks = _json.load(_f)
    emb = np.load(str(Path(ARTIFACTS_DIR) / "embeddings.npy"))
    index = faiss.read_index(str(Path(ARTIFACTS_DIR) / "faiss_index.bin"))
    pretty_print(f"   {len(chunks)} chunks | embeddings {emb.shape} | FAISS {index.ntotal} vectors")
    pretty_print("   Skipping download, chunking, embedding, and index-build cells.")
else:
    pretty_print("⏳ Artifacts not found — will build from scratch.")

✅ Found pre-built artifacts! Loading from disk …
   8509 chunks | embeddings (8509, 768) | FAISS 8509 vectors
   Skipping download, chunking, embedding, and index-build cells.


In [35]:
if not ARTIFACTS_LOADED:
    Path(CORPUS_DIR).mkdir(parents=True, exist_ok=True)

    docs = []  # list of dicts: {title, text, path}

    if DOWNLOAD_FROM_WEB:
        for title, url in GUTENBERG:
            out_path = Path(CORPUS_DIR) / f"{title.replace(' ', '_')}.txt"
            if not out_path.exists():
                pretty_print(f"Downloading: {title}")
                try:
                    r = requests.get(url, timeout=60)
                    r.raise_for_status()
                    clean = strip_gutenberg_boilerplate(r.text)
                    out_path.write_text(clean, encoding="utf-8")
                except Exception as e:
                    pretty_print(f"  Failed ({e}); skipping.")
            else:
                pretty_print(f"Exists: {out_path.name}")
            if out_path.exists():
                docs.append({
                    "title": title,
                    "text": out_path.read_text(encoding="utf-8", errors="ignore"),
                    "path": str(out_path)
                })

    pretty_print(f"Loaded {len(docs)} docs")
else:
    pretty_print("⏩ Skipped (artifacts already loaded)")

⏩ Skipped (artifacts already loaded)


In [36]:
if not ARTIFACTS_LOADED:
    # pretty_print no of words in each document
    for d in docs:
        num_words = len(d["text"].split())
        pretty_print(f"{d['title']}: {num_words} words")
else:
    pretty_print("⏩ Skipped (artifacts already loaded)")

⏩ Skipped (artifacts already loaded)


In [37]:
if not ARTIFACTS_LOADED:
    # --- Chunking ---
    chunks = []  # list of dicts: {id, title, text, preview, source_path, chunk_index}
    for d in docs:
        words = d["text"].split()
        if not words:
            continue
        step = max(1, WORDS_PER_CHUNK - OVERLAP_WORDS)
        idx = 0
        chunk_i = 0
        while idx < len(words):
            segment = words[idx:idx+WORDS_PER_CHUNK]
            if len(segment) < max(60, WORDS_PER_CHUNK//4):
                break
            text_seg = " ".join(segment)
            chunks.append({
                "id": f"{Path(d['title']).name.replace(' ', '_')}#chunk{chunk_i}",
                "title": d["title"],
                "text": text_seg,
                "preview": text_seg[:400],
                "source_path": d["path"],
                "chunk_index": chunk_i,
            })
            chunk_i += 1
            idx += step

    pretty_print(f"Total chunks: {len(chunks)}")
    pd.DataFrame([
        {"id": c["id"], "title": c["title"], "preview": c["preview"][:140] + ("…" if len(c["preview"])>140 else "")}
        for c in chunks[:8]
    ])
else:
    pretty_print(f"⏩ Skipped — {len(chunks)} chunks already loaded from artifacts")

⏩ Skipped — 8509 chunks already loaded from artifacts


In [38]:

## Embed each chunk with your Ollama embedding model (one-by-one API).
## NOTE: Ensure EMBED_MODEL is a valid embeddings model in `ollama list`.
#emb_vectors = []
#for c in tqdm(chunks, desc=f"Embedding with {EMBED_MODEL}"):
#    resp = ollama.embeddings(model=EMBED_MODEL, prompt=c["text"])
#    v = np.asarray(resp["embedding"], dtype="float32")
#    emb_vectors.append(v)

In [39]:
if not ARTIFACTS_LOADED:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    import time
    import numpy as np
    from tqdm import tqdm
    import ollama
    import faiss

    # Tune this based on your machine; 4–8 is usually the sweet spot.
    MAX_WORKERS = 6
    RETRIES = 3
    BACKOFF = 0.6  # seconds, linear backoff

    def embed_text(text: str) -> np.ndarray:
        last_err = None
        for attempt in range(1, RETRIES + 1):
            try:
                resp = ollama.Client(host="http://localhost:11434", trust_env=False).embeddings(model=EMBED_MODEL, prompt=text)
                return np.asarray(resp["embedding"], dtype="float32")
            except Exception as e:
                last_err = e
                if attempt < RETRIES:
                    time.sleep(BACKOFF * attempt)
        raise last_err

    # Parallelize over chunks while preserving original order
    emb_vectors = [None] * len(chunks)
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(embed_text, c["text"]): i for i, c in enumerate(chunks)}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"Embedding ({EMBED_MODEL})"):
            i = futures[fut]
            emb_vectors[i] = fut.result()
else:
    pretty_print(f"⏩ Skipped — embeddings already loaded from artifacts")

⏩ Skipped — embeddings already loaded from artifacts


In [40]:
if not ARTIFACTS_LOADED:
    emb = np.vstack(emb_vectors)  # shape (N, d)
    d = emb.shape[1]
    pretty_print("Embeddings shape:", emb.shape)

    # L2-normalize and build FAISS IP index (IP on unit-norm == cosine similarity)
    emb = l2_normalize(emb)
    index = faiss.IndexFlatIP(d)
    index.add(emb)
    pretty_print("FAISS index size:", index.ntotal)
else:
    pretty_print(f"⏩ Skipped — FAISS index already loaded ({index.ntotal} vectors)")

⏩ Skipped — FAISS index already loaded (8509 vectors)


In [41]:
if not ARTIFACTS_LOADED:
    # === Save all time-consuming artifacts to disk for fast Flask deployment ===
    import json, pickle

    ARTIFACTS_DIR = "rag_artifacts"
    Path(ARTIFACTS_DIR).mkdir(parents=True, exist_ok=True)

    # 1. Save chunks metadata as JSON
    with open(Path(ARTIFACTS_DIR) / "chunks.json", "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False)
    pretty_print(f"Saved {len(chunks)} chunks to {ARTIFACTS_DIR}/chunks.json")

    # 2. Save normalized embeddings as numpy array
    np.save(Path(ARTIFACTS_DIR) / "embeddings.npy", emb)
    pretty_print(f"Saved embeddings shape {emb.shape} to {ARTIFACTS_DIR}/embeddings.npy")

    # 3. Save FAISS index
    faiss.write_index(index, str(Path(ARTIFACTS_DIR) / "faiss_index.bin"))
    pretty_print(f"Saved FAISS index ({index.ntotal} vectors) to {ARTIFACTS_DIR}/faiss_index.bin")

    # 4. Save config so Flask app knows model name, chunk params, etc.
    config = {
        "EMBED_MODEL": EMBED_MODEL,
        "WORDS_PER_CHUNK": WORDS_PER_CHUNK,
        "OVERLAP_WORDS": OVERLAP_WORDS,
        "TOPK": TOPK,
    }
    with open(Path(ARTIFACTS_DIR) / "config.json", "w") as f:
        json.dump(config, f, indent=2)
    pretty_print(f"Saved config to {ARTIFACTS_DIR}/config.json")

    pretty_print("\n✅ All artifacts saved!")
else:
    pretty_print("⏩ Skipped — artifacts already exist on disk")

⏩ Skipped — artifacts already exist on disk


In [42]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

## How `expand_with_neighbors_simple()` works

The function takes the FAISS hits and, for each one, also grabs the chunks **immediately before and after** it in the same document. Those extra chunks give the LLM more surrounding context than a single 300-word window can.

### Inputs

- **`I`** — indices array of shape `(1, num_hits)` returned by FAISS; `I[0]` is the list of matched chunk indices in relevance order.
- **`chunks`** — the full list of chunk dicts (text + metadata).
- **`neighbors`** — how many chunks on each side to include (e.g. `1` means `[-1, 0, +1]`).
- **`max_out`** — safety cap on the total number of chunks returned.

### The algorithm

1. **For each ANN hit** (best match first):
2. **For each offset** in `[-neighbors, …, +neighbors]`:
   - Look up `(same document, hit.chunk_index + offset)` in `KEY_TO_IDX`.
   - If it exists and we haven't added it yet, append it to the output.
3. **Stop** once we've collected `max_out` chunks.
4. **Return** the chunks as a flat list — each chunk is its own context entry.

### Example

- FAISS hit: chunk `42` of *Pride and Prejudice*, `neighbors=1`
- Output: chunks `41`, `42`, `43` — three separate context entries.

If two different hits happen to share a neighbor (e.g. hit-A is chunk 42 and hit-B is chunk 44, both want chunk 43), the `seen` set deduplicates so chunk 43 only appears once.

### Why the `KEY_TO_IDX` lookup map?

```python
KEY_TO_IDX = {(source_path, chunk_index): global_idx for ...}
```

Given a chunk's `(document, chunk_number)`, we get its global index in `chunks` in **O(1)**. Without it, we'd have to scan all chunks every time we wanted a neighbor — slow once you have thousands of chunks.


## How `search_windowed()` works

This is the top-level search function. It takes a user query and returns:
1. **`df_hits`** — a DataFrame of the raw FAISS hits (for inspection / debugging).
2. **`contexts`** — the expanded list of chunks ready to feed to an LLM.

### Inputs

- **`query`** — natural-language question / search string.
- **`topk`** — how many FAISS hits to retrieve (default `TOPK = 5`).
- **`max_out`** — maximum chunks in the final `contexts` list (default `8`).

### Step 1 — Embed the query

```python
q_vec = ollama.Client(host="http://localhost:11434", trust_env=False) \
              .embeddings(model=EMBED_MODEL, prompt=query)["embedding"]
q_vec = q_vec / (np.linalg.norm(q_vec) + 1e-12)   # L2-normalize
```

We use the **explicit Client** here (same pattern as the Ollama quickstart at the top) so that a VPN / proxy on the machine doesn't intercept the localhost call. Normalizing to unit length means a dot product against the FAISS index is exactly cosine similarity.

### Step 2 — FAISS top-k search

```python
D, I = index.search(q_vec.reshape(1, -1), topk)
```

- `I[0]` → the indices of the top-k most similar chunks (best first).
- `D[0]` → their similarity scores (higher = more similar, since we used `IndexFlatIP` on unit-norm vectors).

### Step 3 — Pick a neighbor width

```python
neighbors = 1 if WORDS_PER_CHUNK >= 260 else 2
```

With large 300-word chunks each hit already has plenty of context, so `±1` neighbors is enough. Shrink the chunks and you'll want a wider window.

### Step 4 — Confidence / margin heuristic

We retrieved `topk` hits, but we don't always want to expand all of them. Two signals tell us how much to trust the top match:

```python
top1   = D[0][0]
top2   = D[0][1]
margin = top1 - top2
init_topk = 1 if (top1 >= 0.35 and margin >= 0.05) else min(3, topk)
```

- **`top1 >= 0.35`** — the best match is similar enough to the query that we believe it (cosine similarity threshold).
- **`margin >= 0.05`** — the best match clearly beats the runner-up; the answer isn't ambiguous.

**Both conditions met → expand only the top 1 hit.** Otherwise the results look ambiguous, so cast a wider net and expand the top 3.

Intuition:

| Query | top1 | top2 | margin | Outcome |
|---|---|---|---|---|
| "Who is Sherlock Holmes?" | 0.89 | 0.70 | 0.19 | Confident → 1 hit |
| "Victorian clothing" | 0.75 | 0.72 | 0.03 | Ambiguous → 3 hits |
| "What is photosynthesis?" *(not in corpus)* | 0.22 | 0.20 | 0.02 | Weak → 3 hits |

### Step 5 — Expand with neighbors

```python
I_init   = np.array([I[0][:init_topk]])
contexts = expand_with_neighbors_simple(I_init, chunks, neighbors=neighbors, max_out=max_out)
```

Take only the chosen `init_topk` hits and pass them to the function from the previous markdown cell. Each hit contributes itself plus its `±neighbors` chunks, deduplicated.

### Step 6 — Build the `df_hits` table

For each of the original `topk` hits we record `score`, `id`, `title`, `chunk_index`, and a short text preview, then wrap it in a DataFrame. This is purely for **looking at what the retriever found** — it's not what the LLM sees.

### Step 7 — Return

```python
return df_hits, contexts
```

- `df_hits` → for you, the developer.
- `contexts` → for the LLM (or whatever consumes the retrieval result).


## Visual System Flow

```
┌─────────────────────────────────────────────┐
│  USER QUERY                                 │
│  "Elizabeth's love for Darcy"               │
└───────────────────┬─────────────────────────┘
                    │
                    ▼
          ┌──────────────────────┐
          │  Embed Query         │
          │  (Ollama)            │
          │  → 1 vector          │
          └─────────┬────────────┘
                    │
                    ▼
          ┌──────────────────────────────────┐
          │  FAISS top-k search              │
          │  I = [42, 108, 33, 5, 71]        │
          │  D = [0.87, 0.65, 0.58, ...]     │
          └─────────┬────────────────────────┘
                    │
        ┌───────────┴───────────────┐
        │                           │
        ▼                           ▼
┌─────────────────┐   ┌────────────────────────────┐
│  df_hits table  │   │  Expand each hit with      │
│  (raw hits)     │   │  ±neighbor chunks          │
└─────────────────┘   │  (same document, dedupe)   │
                      └─────────┬──────────────────┘
                                │
                                ▼
                      ┌─────────────────────┐
                      │  contexts (list)    │
                      │  → fed to the LLM   │
                      └─────────────────────┘
```

---

## Why two functions?

- **`expand_with_neighbors_simple()`** — the mechanic: given some hit indices, return those chunks + their neighbors.
- **`search_windowed()`** — the orchestrator: embed → search → expand → package the result.

Separating them means you can swap retrieval strategies (different `topk`, different `neighbors`, an entirely different expansion rule) without touching the other piece.

---

## Why fetch neighbors at all?

A chunk is just a 300-word window — it may cut a sentence in half or stop right before the most relevant sentence. Including the chunk **before** and **after** gives the LLM enough surrounding text to actually answer the question.

```
Without neighbors:  [chunk 42]                   ← may miss context
With ±1 neighbors:  [chunk 41][chunk 42][chunk 43] ← much more context
```

---

## Example walkthrough

**Query:** `"What is the white whale in Moby Dick?"`

1. **Embed** the query → 1 vector.
2. **FAISS search**, top-5 → `I = [17, 45, 12, 88, 3]`, `D = [0.91, 0.58, 0.52, 0.49, 0.44]`.
3. **Neighbor width** → `WORDS_PER_CHUNK = 300 ≥ 260`, so `neighbors = 1`.
4. **Expand**:
   - Hit 17 → adds chunks 16, 17, 18.
   - Hit 45 → adds chunks 44, 45, 46.
   - Hit 12 → adds chunks 11, 12, 13.
   - …stop when we hit `max_out = 8`.
5. **Return** `df_hits` (all 5 raw matches) + `contexts` (≤ 8 chunks).


In [43]:
query = "Best detective in the world"
topk = 5

q_vec = np.asarray(ollama.Client(host="http://localhost:11434", trust_env=False).embeddings(model=EMBED_MODEL, prompt=query)["embedding"], dtype="float32")
q_vec = q_vec / (np.linalg.norm(q_vec) + 1e-12)
D, I = index.search(q_vec.reshape(1, -1), topk)

D.shape

(1, 5)

In [44]:

print(f"\nTop {topk} hits for query: '{query}'\n")
for rank, (dist, idx) in enumerate(zip(D[0], I[0]), start=1):
    chunk = chunks[idx]
    pretty_print(f"Rank {rank} | Score {dist:.4f} | {chunk['title']} (chunk {chunk['chunk_index']})")
    pretty_print("  ", chunk["preview"])
    print()


Top 5 hits for query: 'Best detective in the world'

Rank 1 | Score 0.3802 | Adventures of Sherlock Holmes (chunk 134)
   meet him at Boscombe Pool was someone who had been in Australia.” “What of
the rat, then?” Sherlock Holmes took a folded paper from his pocket and
flattened it out on the table. “This is a map of the Colony of Victoria,” he
said. “I wired to Bristol for it last night.” He put his hand over part of the
map. “What do you read?” “ARAT,” I read. “And now?” He raised his hand.
“BALLARAT.” “Quite so. Th

Rank 2 | Score 0.3763 | Adventures of Sherlock Holmes (chunk 216)
   I continue to retain the hat of the unknown gentleman who lost his Christmas
dinner.” “Did he not advertise?” “No.” “Then, what clue could you have as to his
identity?” “Only as much as we can deduce.” “From his hat?” “Precisely.” “But
you are joking. What can you gather from this old battered felt?” “Here is my
lens. You know my methods. What can you gather yourself as to the individuality
of the

Rank

In [45]:
# --- Simple windowed retrieval: expand each ANN hit with ±neighbor chunks ---


# Lookup map for (doc, chunk_index) -> global idx
KEY_TO_IDX = {(c["source_path"], c["chunk_index"]): gi for gi, c in enumerate(chunks)}


def expand_with_neighbors_simple(I, chunks, neighbors=1, max_out=8):
    """
    For each ANN hit in I[0], include the hit itself plus its ±neighbor
    chunks from the same document. Dedupe, preserve hit order, and stop
    once we've collected `max_out` chunks.

    Returns a flat list of context dicts — one per chunk (no merging).
    """
    seen = set()
    contexts = []

    for gi in I[0]:
        c = chunks[int(gi)]
        doc = c["source_path"]
        ci  = c["chunk_index"]
        # the hit itself + each neighbor
        for delta in range(-neighbors, neighbors + 1):
            j = KEY_TO_IDX.get((doc, ci + delta))
            if j is None or j in seen:
                continue
            seen.add(j)
            n = chunks[j]
            contexts.append({
                "title": n["title"],
                "source_path": n["source_path"],
                "chunk_index": n["chunk_index"],
                "text": n["text"],
                "approx_words": len(n["text"].split()),
            })
            if len(contexts) >= max_out:
                return contexts
    return contexts


def search_windowed(query: str, topk: int = TOPK, max_out: int = 8):
    """
    1) Embed the query (Ollama)
    2) ANN search the FAISS index for top-k chunks
    3) Decide how many of those hits to actually expand, using a simple
       confidence/margin heuristic on the FAISS scores
    4) Expand each chosen hit with ±neighbor chunks from the same document
    Returns: (df_hits, contexts)
    """
    q_vec = np.asarray(
        ollama.Client(host="http://localhost:11434", trust_env=False)
              .embeddings(model=EMBED_MODEL, prompt=query)["embedding"],
        dtype="float32",
    )
    q_vec = q_vec / (np.linalg.norm(q_vec) + 1e-12)
    D, I = index.search(q_vec.reshape(1, -1), topk)

    # ±1 neighbors for ~300-word chunks; widen if chunks are shorter
    neighbors = 1 if WORDS_PER_CHUNK >= 260 else 2

    # Confidence/margin heuristic: how many top hits should we actually expand?
    # - If the best match is strong (>= 0.35) AND clearly beats #2 (margin >= 0.05),
    #   we trust just that one hit.
    # - Otherwise the results look ambiguous, so cast a wider net (up to 3).
    top1   = float(D[0][0])
    top2   = float(D[0][1]) if len(D[0]) > 1 else 0.0
    margin = top1 - top2
    init_topk = 1 if (top1 >= 0.35 and margin >= 0.05) else min(3, topk)

    I_init = np.array([I[0][:init_topk]])

    contexts = expand_with_neighbors_simple(I_init, chunks, neighbors=neighbors, max_out=max_out)

    # Table of raw ANN hits (for transparency / debugging)
    rows = []
    for score, idx_ in zip(D[0].tolist(), I[0].tolist()):
        m = chunks[idx_]
        rows.append({
            "score": round(score, 3),
            "id": m["id"],
            "title": m["title"],
            "source_path": m["source_path"],
            "chunk_index": m["chunk_index"],
            "preview": (m["preview"][:220] + "…") if len(m["preview"]) > 220 else m["preview"],
        })
    df_hits = pd.DataFrame(rows)
    return df_hits, contexts


In [46]:
# --- Quick test query ---
QUERY = "Elizabeth Bennet's changing feelings for Mr. Darcy"
df_hits, contexts = search_windowed(QUERY, topk=TOPK, max_out=8)

print("=== Initial ANN Hits ===")
display(df_hits)

print("\n=== Expanded Contexts (hits + ±neighbors) ===")
display(pd.DataFrame([{
    "title": c["title"],
    "source": Path(c["source_path"]).name,
    "chunk_index": c["chunk_index"],
    "approx_words": c["approx_words"],
    "preview": (c["text"][:220] + "…") if len(c["text"]) > 220 else c["text"],
} for c in contexts]))

=== Initial ANN Hits ===


,score,id,title,source_path,chunk_index,preview
0,0.517,Pride_and_Prejudice#chunk457,Pride and Prejudice,corpus_jupyter/Pride_and_Prejudice.txt,457,"Mr. Darcy!--and so it does, I vow. Well, any f..."
1,0.509,Pride_and_Prejudice#chunk83,Pride and Prejudice,corpus_jupyter/Pride_and_Prejudice.txt,83,"employed, Elizabeth could not help observing, ..."
2,0.490,Pride_and_Prejudice#chunk518,Pride and Prejudice,corpus_jupyter/Pride_and_Prejudice.txt,518,good as a lord! And a special licence--you mus...
3,0.486,Pride_and_Prejudice#chunk349,Pride and Prejudice,corpus_jupyter/Pride_and_Prejudice.txt,349,they are! He takes them now for people of fash...
4,0.480,Pride_and_Prejudice#chunk357,Pride and Prejudice,corpus_jupyter/Pride_and_Prejudice.txt,357,who had expected to find in her as acute and u...



=== Expanded Contexts (hits + ±neighbors) ===


,title,source,chunk_index,approx_words,preview
0,Pride and Prejudice,Pride_and_Prejudice.txt,456,300,"but she does not know, no one can know, how mu..."
1,Pride and Prejudice,Pride_and_Prejudice.txt,457,300,"Mr. Darcy!--and so it does, I vow. Well, any f..."
2,Pride and Prejudice,Pride_and_Prejudice.txt,458,300,"again, was almost equal to what she had known ..."
3,Pride and Prejudice,Pride_and_Prejudice.txt,82,300,"Darcy were not such a great tall fellow, in co..."
4,Pride and Prejudice,Pride_and_Prejudice.txt,83,300,"employed, Elizabeth could not help observing, ..."
5,Pride and Prejudice,Pride_and_Prejudice.txt,84,300,to dance a reel at all; and now despise me if ...
6,Pride and Prejudice,Pride_and_Prejudice.txt,517,300,"utter a syllable. Nor was it under many, many ..."
7,Pride and Prejudice,Pride_and_Prejudice.txt,518,300,good as a lord! And a special licence--you mus...
